In [34]:
import pandas as pd
import pandas_ta_classic as ta
import config
from oandapyV20 import API
import oandapyV20.endpoints.instruments as instruments
import oandapyV20.endpoints.orders as orders

In [35]:
#Set up OANDA connection
client = API(access_token=config.OANDA_API_KEY)

In [36]:
#define variables
timeframe = "M1"
instrument = "USD_JPY"

In [37]:
# Get latest Candle Data
def get_Candles(tf):

    params = {
        "granularity": tf,
        "price": "A"
    }

    r = instruments.InstrumentsCandles(instrument=instrument, params=params)
    candles = client.request(r)['candles']
    candles

    #Convert to Panda's DataFrame
    data = []

    for c in candles:
        if c["complete"]:
            data.append({
                "time": c["time"],
                "open": float(c["ask"]["o"]),
                "high": float(c["ask"]["h"]),
                "low": float(c["ask"]["l"]),
                "close": float(c["ask"]["c"])
            })
    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    return df

In [38]:
# Function: Return EMA Indicators & ATR Stop Loss
def get_indicators(df, high_ind, low_ind):

    #EMAs
    df["EMA_Low"] = ta.ema(df["close"], length=low_ind)
    df["EMA_High"] = ta.ema(df["close"], length=high_ind)

    # ATR Stop_Loss
    df["ATR_14"] = ta.atr(df["high"], df["low"], df["close"], length=14)

    return df


In [39]:
# Function: Place Order
def place_order(stop_loss, take_profit):
    data = {
        "order": {
            "instrument": instrument,
            "units": 1, 
            "type": "MARKET",
            "stopLossOnFill": {"price": f"{stop_loss:.3f}"},
            "takeProfitOnFill": {"price": f"{take_profit:.3f}"}
        }
    }
    r = orders.OrderCreate(config.OANDA_ACCOUNT_ID, data=data)
    client.request(r)
    print(f"Placed order for{instrument} with stop loss at {round(stop_loss, 3)} and take profit at {round(take_profit, 3)}")

In [ ]:
# Funtion: EMA Crossover logic
def ema_crossover(df):
    #Take profit ration 1.5%
    tp_ratio = 1.5

    last_candle = df.iloc[-1]
    previous_candle = df.iloc[-2]

    if last_candle['EMA_Low'] > last_candle['EMA_High'] and previous_candle['EMA_Low'] < previous_candle['EMA_High']:
        print("Crossover Buy Signal is generated")

        #Calculate stop loss
        entry_price = last_candle['close']
        stop_loss = entry_price - last_candle['ATR_14']

        #Calculate take profit
        stop_distence = entry_price - stop_loss
        take_profit = entry_price + (stop_distence * tp_ratio)

        #Place Order
        place_order(stop_loss=stop_loss, take_profit=take_profit)
    else:
        print ("Crossover Strategy conditions not met")
        print (f"Last Candle: 5: {last_candle['EMA_Low']} 8: {last_candle['EMA_High']} and Previous Candle: 5: {previous_candle['EMA_Low']} 8: {previous_candle['EMA_High']}")
    

In [41]:
#

In [42]:
def run_bot():
    print("Starting TradePI Bot ......")
    price = get_Candles(timeframe)
    price = get_indicators(price, high_ind=8, low_ind=5)
    ema_crossover(price)
    


In [49]:
run_bot()

Starting TradePI Bot ......
Crossover Strategy conditions not met
Last Candle: 163.2254651711844 !> 163.22026370988996 and Previous Candle 163.2206977567766 !< 163.21605334128708
